# Notebook 3: Multi-Output LSTM — GHG Forecasting

Implements an LSTM model for sequential GHG forecasting with LOOCV evaluation.

**Architecture**: 2-layer LSTM (64 → 32 units) with Dropout, trained with Huber loss

**Targets**: `co2`, `methane`, `nitrous_oxide`, `oil_co2`, `gas_co2`, `coal_co2`

**Sequence length**: 6 years of historical data used to predict the next year

## Results Summary (LOOCV)
| Target | R² | MAPE |
|--------|----|------|
| CO2 | 0.96 | ~4.3% |
| Methane | — | — |
| Nitrous Oxide | — | — |

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import LeaveOneOut

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import Huber

## 2. Load Data & Feature Engineering

In [ ]:
df = pd.read_csv('data/merged_co2_energy_dataset.csv')

exclude_cols = [
    'co2_per_unit_energy', 'co2_growth_prct', 'co2_per_gdp', 'flaring_co2',
    'flaring_co2_per_capita', 'co2_per_capita', 'methane_per_capita', 'primary_energy_consumption'
]
df.drop(columns=exclude_cols, inplace=True, errors='ignore')
df = df.sort_values('year')

# Composite cumulative GHG feature
df['cumulative_ghg'] = (
    df['methane'].cumsum() +
    df['co2'].cumsum() +
    df['nitrous_oxide'].cumsum()
)

features = ['Diesel', 'Kerosene', 'Crude Oil Production', 'Gasoline', 'oil_co2', 'cumulative_ghg']
targets  = ['co2', 'methane', 'nitrous_oxide', 'oil_co2', 'gas_co2', 'coal_co2']

data = df[features + targets].dropna()
print(f"Dataset shape: {data.shape}")

## 3. Normalisation & Sequence Preparation

MinMaxScaler normalises all features and targets to [0, 1].
Sequences of length 6 are created: each sample uses 6 consecutive years to predict year 7.

In [ ]:
scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(data)

SEQ_LEN = 6
X_all, y_all = [], []

for i in range(SEQ_LEN, len(data_scaled)):
    X_all.append(data_scaled[i - SEQ_LEN:i, :len(features)])
    y_all.append(data_scaled[i, len(features):])

X_all = np.array(X_all)
y_all = np.array(y_all)

print(f"X shape: {X_all.shape}  (samples, seq_len, features)")
print(f"y shape: {y_all.shape}  (samples, targets)")

## 4. Evaluation Metrics

In [ ]:
def smape(y_true, y_pred):
    return 100 / len(y_true) * np.sum(
        2 * np.abs(y_pred - y_true) / (np.abs(y_pred) + np.abs(y_true) + 1e-8)
    )

def mape(y_true, y_pred):
    return 100 * np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8)))

## 5. Model Architecture

In [ ]:
def build_model(input_shape, output_size):
    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        LSTM(32),
        Dense(output_size)
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss=Huber())
    return model

# Preview architecture
preview_model = build_model((X_all.shape[1], X_all.shape[2]), y_all.shape[1])
preview_model.summary()

## 6. LOOCV Training

> Note: LOOCV with LSTM is computationally expensive. Each fold trains a fresh model for 150 epochs.

In [ ]:
loo = LeaveOneOut()
preds, trues = [], []

for fold, (train_idx, test_idx) in enumerate(loo.split(X_all)):
    X_train, X_test = X_all[train_idx], X_all[test_idx]
    y_train, y_test = y_all[train_idx], y_all[test_idx]

    model = build_model((X_all.shape[1], X_all.shape[2]), y_all.shape[1])
    model.fit(X_train, y_train, epochs=150, batch_size=2, verbose=0)

    preds.append(model.predict(X_test, verbose=0)[0])
    trues.append(y_test[0])

    if (fold + 1) % 5 == 0:
        print(f"Fold {fold + 1}/{len(X_all)} complete")

preds = np.array(preds)
trues = np.array(trues)
print("LOOCV training complete.")

## 7. Inverse Transform & Evaluate

In [ ]:
# Inverse transform — reconstruct full feature+target array for scaler
X_last = X_all[:, -1]  # last timestep of each sequence
inv_pred = scaler.inverse_transform(np.concatenate([X_last[:len(preds)], preds], axis=1))
inv_true = scaler.inverse_transform(np.concatenate([X_last[:len(trues)], trues], axis=1))

print("[LSTM LOOCV Evaluation]")
for i, target in enumerate(targets):
    idx = len(features) + i
    y_true = inv_true[:, idx]
    y_pred = inv_pred[:, idx]
    print(f"\n--- {target.upper()} ---")
    print(f"  R²:    {r2_score(y_true, y_pred):.5f}")
    print(f"  MAE:   {mean_absolute_error(y_true, y_pred):.5f}")
    print(f"  MSE:   {mean_squared_error(y_true, y_pred):.5f}")
    print(f"  MAPE:  {mape(y_true, y_pred):.3f}%")
    print(f"  sMAPE: {smape(y_true, y_pred):.3f}%")

## 8. Visualisation — Predicted vs Actual & Residual Distribution

In [ ]:
os.makedirs('outputs', exist_ok=True)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, target in enumerate(targets):
    idx = len(features) + i
    actual    = inv_true[:, idx]
    predicted = inv_pred[:, idx]

    axes[i].scatter(actual, predicted, alpha=0.7, color='steelblue', label='Predictions')
    axes[i].plot([actual.min(), actual.max()], [actual.min(), actual.max()], 'r--', label='Perfect fit')
    axes[i].set_title(f"{target.upper()} — Actual vs Predicted")
    axes[i].set_xlabel('Actual')
    axes[i].set_ylabel('Predicted')
    axes[i].legend()

plt.tight_layout()
plt.savefig('outputs/lstm_predicted_vs_actual.png', dpi=150)
plt.show()

# Residual distributions with 95% CI
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, target in enumerate(targets):
    idx = len(features) + i
    residuals = inv_pred[:, idx] - inv_true[:, idx]
    ci = 1.96 * np.std(residuals)

    sns.histplot(residuals, kde=True, ax=axes[i], stat='density', color='skyblue')
    axes[i].axvline(-ci, color='red', linestyle='--', label=f'95% CI: ±{ci:.2f}')
    axes[i].axvline( ci, color='red', linestyle='--')
    axes[i].set_title(f"{target.upper()} — Residual Distribution")
    axes[i].legend()

plt.tight_layout()
plt.savefig('outputs/lstm_residual_distributions.png', dpi=150)
plt.show()